# 02_eval_parser_adapter_stratified

Eval-only notebook cho Government AI Parser.

1. Detect dataset + final LoRA adapter
2. Load Qwen3-4B + adapter
3. Stratified sample validation/test
4. Evaluate JSON/schema/intent/family/slots/dangerous wrong-query
5. Xuất reports kiểm tra

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.51.0",
    "accelerate",
    "peft",
    "bitsandbytes",
    "jsonschema",
    "protobuf>=5.29.1,<6",
    "sentencepiece",
])

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "wandb"], check=False)

import gc
import time
import math
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from jsonschema import Draft7Validator

print("Setup OK", flush=True)
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count(), flush=True)

In [ ]:
BASE_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

EVAL_MAX_SAMPLES_PER_SPLIT = 1000
EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 192
RANDOM_SEED = 42

KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/government-parser-eval")

RAW_DATA_DIR = WORK_DIR / "data" / "raw"
CONFIG_DIR = WORK_DIR / "configs"
MODEL_DIR = WORK_DIR / "model"
REPORT_DIR = WORK_DIR / "reports"
EXPORT_DIR = WORK_DIR / "export"

for p in [RAW_DATA_DIR, CONFIG_DIR, MODEL_DIR, REPORT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def find_first_file(filename: str):
    matches = list(KAGGLE_INPUT_DIR.rglob(filename))
    return matches[0] if matches else None

def find_dataset_root():
    candidates = []

    for validation_path in KAGGLE_INPUT_DIR.rglob("parser_validation.v1.jsonl"):
        root = validation_path.parent
        score = 0

        if "government-parser-dataset-v1" in str(root):
            score += 100
        if (root / "parsed_query_schema.v1.json").exists():
            score += 50
        if (root / "parser_intents.v1.json").exists():
            score += 10
        if (root / "question_families.v1.json").exists():
            score += 10

        if "/notebooks/" in str(root):
            score -= 20

        candidates.append((score, root))

    if not candidates:
        raise FileNotFoundError("Không tìm thấy parser_validation.v1.jsonl trong /kaggle/input")

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("Dataset root candidates:")
    for score, root in candidates:
        print(score, root)

    return candidates[0][1]

SOURCE_DATASET_DIR = find_dataset_root()

validation_src = SOURCE_DATASET_DIR / "parser_validation.v1.jsonl"
test_src = SOURCE_DATASET_DIR / "parser_test.v1.jsonl"
schema_src = SOURCE_DATASET_DIR / "parsed_query_schema.v1.json"

print("Selected SOURCE_DATASET_DIR:", SOURCE_DATASET_DIR)

if validation_src is None or test_src is None or schema_src is None:
    raise FileNotFoundError("Thiếu dataset/config gốc. Hãy attach government-parser-dataset-v1 vào notebook.")

SOURCE_DATASET_DIR = validation_src.parent

required_data = ["parser_validation.v1.jsonl", "parser_test.v1.jsonl"]
required_configs = [
    "parsed_query_schema.v1.json",
    "parser_intents.v1.json",
    "parser_enums.v1.json",
    "question_families.v1.json",
    "country_catalog.v1.json",
    "indicator_catalog.v1.json",
    "analytics_metadata.v1.json",
]

for fn in required_data:
    shutil.copy2(SOURCE_DATASET_DIR / fn, RAW_DATA_DIR / fn)

for fn in required_configs:
    src = SOURCE_DATASET_DIR / fn
    if src.exists():
        shutil.copy2(src, CONFIG_DIR / fn)
    else:
        print("Warning missing config:", src, flush=True)

RAW_VALIDATION_PATH = RAW_DATA_DIR / "parser_validation.v1.jsonl"
RAW_TEST_PATH = RAW_DATA_DIR / "parser_test.v1.jsonl"

SCHEMA_PATH = CONFIG_DIR / "parsed_query_schema.v1.json"
INTENTS_PATH = CONFIG_DIR / "parser_intents.v1.json"
ENUMS_PATH = CONFIG_DIR / "parser_enums.v1.json"
QUESTION_FAMILIES_PATH = CONFIG_DIR / "question_families.v1.json"
COUNTRY_CATALOG_PATH = CONFIG_DIR / "country_catalog.v1.json"
INDICATOR_CATALOG_PATH = CONFIG_DIR / "indicator_catalog.v1.json"

def find_adapter_dirs():
    out = []
    for cfg in KAGGLE_INPUT_DIR.rglob("adapter_config.json"):
        d = cfg.parent
        if (d / "adapter_model.safetensors").exists():
            out.append(d)
    return out

adapter_dirs = find_adapter_dirs()
FINAL_ADAPTER_DIR = MODEL_DIR / "final_adapter"

if FINAL_ADAPTER_DIR.exists():
    shutil.rmtree(FINAL_ADAPTER_DIR)

if adapter_dirs:
    # Ưu tiên path có final_adapter/parser_qwen nếu có.
    adapter_dirs = sorted(adapter_dirs, key=lambda p: (("final" not in str(p).lower()) and ("adapter" not in str(p).lower()), len(str(p))))
    SOURCE_ADAPTER_DIR = adapter_dirs[0]
    shutil.copytree(SOURCE_ADAPTER_DIR, FINAL_ADAPTER_DIR)
else:
    zip_candidates = list(KAGGLE_INPUT_DIR.rglob("government_parser_qwen3_4b_lora_artifact.zip"))
    if not zip_candidates:
        raise FileNotFoundError("Không tìm thấy final adapter folder hoặc artifact zip trong /kaggle/input.")
    artifact_zip = zip_candidates[0]
    extract_dir = MODEL_DIR / "artifact_extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(artifact_zip, "r") as z:
        z.extractall(extract_dir)
    extracted = []
    for cfg in extract_dir.rglob("adapter_config.json"):
        d = cfg.parent
        if (d / "adapter_model.safetensors").exists():
            extracted.append(d)
    if not extracted:
        raise FileNotFoundError("Artifact zip không chứa adapter hợp lệ.")
    SOURCE_ADAPTER_DIR = extracted[0]
    shutil.copytree(SOURCE_ADAPTER_DIR, FINAL_ADAPTER_DIR)

print("SOURCE_DATASET_DIR:", SOURCE_DATASET_DIR, flush=True)
print("SOURCE_ADAPTER_DIR:", SOURCE_ADAPTER_DIR, flush=True)
print("FINAL_ADAPTER_DIR:", FINAL_ADAPTER_DIR, flush=True)
print("Adapter files:", sorted([p.name for p in FINAL_ADAPTER_DIR.iterdir()]), flush=True)

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

SCHEMA = load_json(SCHEMA_PATH)
INTENTS = set(load_json(INTENTS_PATH))
ENUMS = load_json(ENUMS_PATH)
QUESTION_FAMILIES = load_json(QUESTION_FAMILIES_PATH)
COUNTRY_CATALOG = load_json(COUNTRY_CATALOG_PATH)
INDICATOR_CATALOG = load_json(INDICATOR_CATALOG_PATH)

FAMILY_TO_INTENT = {x["id"]: x["intent"] for x in QUESTION_FAMILIES["families"]}
COUNTRY_CODES = {x["code"] for x in COUNTRY_CATALOG["countries"]}
INDICATOR_CODES = {x["code"] for x in INDICATOR_CATALOG["indicators"]}
ALLOWED_GROUPS = set(ENUMS.get("country_groups", []))

validator = Draft7Validator(SCHEMA)

validation_rows = load_jsonl(RAW_VALIDATION_PATH)
test_rows = load_jsonl(RAW_TEST_PATH)

print("Validation rows:", len(validation_rows), flush=True)
print("Test rows:", len(test_rows), flush=True)
print("Intents:", len(INTENTS), "Families:", len(FAMILY_TO_INTENT), flush=True)
print("Countries:", len(COUNTRY_CODES), "Indicators:", len(INDICATOR_CODES), flush=True)

In [ ]:
def stratified_sample(rows, n=None, seed=42):
    if n is None or n >= len(rows):
        return list(rows)

    rng = random.Random(seed)

    groups = defaultdict(list)
    for row in rows:
        key = (
            row.get("generation_source"),
            row.get("intent"),
            row.get("language_style"),
        )
        groups[key].append(row)

    selected = []
    group_items = list(groups.items())
    rng.shuffle(group_items)

    for key, items in group_items:
        if len(selected) < n:
            selected.append(rng.choice(items))

    remaining = n - len(selected)
    if remaining > 0:
        # proportional pool
        pool = []
        for key, items in group_items:
            items_copy = list(items)
            rng.shuffle(items_copy)
            pool.extend(items_copy)
        rng.shuffle(pool)

        seen_ids = {x.get("sample_id") for x in selected}
        for item in pool:
            if len(selected) >= n:
                break
            sid = item.get("sample_id")
            if sid not in seen_ids:
                selected.append(item)
                seen_ids.add(sid)

    rng.shuffle(selected)
    return selected[:n]

eval_validation_rows = stratified_sample(validation_rows, EVAL_MAX_SAMPLES_PER_SPLIT, RANDOM_SEED)
eval_test_rows = stratified_sample(test_rows, EVAL_MAX_SAMPLES_PER_SPLIT, RANDOM_SEED + 1)

def summarize_rows(name, rows):
    print("=" * 80, flush=True)
    print(name, "rows:", len(rows), flush=True)
    print("generation_source:", dict(Counter(r.get("generation_source") for r in rows)), flush=True)
    print("language_style:", dict(Counter(r.get("language_style") for r in rows)), flush=True)
    print("top intents:", Counter(r.get("intent") for r in rows).most_common(10), flush=True)

summarize_rows("validation_sample", eval_validation_rows)
summarize_rows("test_sample", eval_test_rows)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_ADAPTER_DIR),
    use_fast=True,
    trust_remote_code=False,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[START] Load base model", flush=True)
t0 = time.time()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    attn_implementation="sdpa",
)

print("[DONE] Load base model seconds:", round(time.time() - t0, 2), flush=True)

print("[START] Load LoRA adapter", flush=True)
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR))
model.eval()
model.config.use_cache = True

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

print("[DONE] Model ready", flush=True)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.3f}GB "
        f"reserved={torch.cuda.memory_reserved(i)/1024**3:.3f}GB",
        flush=True,
    )

In [ ]:
def safe_json_loads(text: str):
    text = (text or "").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)

def validate_parsed_query(parsed: dict):
    errors = []

    for e in sorted(validator.iter_errors(parsed), key=lambda x: x.path):
        path = ".".join(str(p) for p in e.path)
        errors.append(f"schema_error:{path}:{e.message}")

    intent = parsed.get("intent")
    family = parsed.get("question_family")

    if intent not in INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if family not in FAMILY_TO_INTENT:
        errors.append(f"invalid_question_family:{family}")
    elif intent != FAMILY_TO_INTENT[family]:
        errors.append(f"family_intent_mismatch:{family}:parsed={intent}:expected={FAMILY_TO_INTENT[family]}")

    for x in parsed.get("indicators", []) or []:
        if x not in INDICATOR_CODES:
            errors.append(f"invalid_indicator:{x}")

    for x in parsed.get("countries", []) or []:
        if x not in COUNTRY_CODES:
            errors.append(f"invalid_country:{x}")

    for x in parsed.get("country_groups", []) or []:
        if x not in ALLOWED_GROUPS:
            errors.append(f"invalid_country_group:{x}")

    for field in ["ranking_order", "chart_preference", "aggregation", "relative_time", "event_time"]:
        allowed = set(ENUMS.get(field, []))
        if parsed.get(field) not in allowed:
            errors.append(f"invalid_enum:{field}:{parsed.get(field)}")

    return errors

def list_micro_counts(pred, gold):
    pred = set(pred or [])
    gold = set(gold or [])
    return len(pred & gold), len(pred - gold), len(gold - pred)

def is_executable(parsed):
    if not parsed:
        return False
    if parsed.get("intent") in {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}:
        return False
    if parsed.get("needs_clarification") is True:
        return False
    return True

CRITICAL_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
]

def is_dangerous_wrong(pred, gold, schema_ok):
    if not schema_ok or not is_executable(pred):
        return False

    if not is_executable(gold):
        return True

    for field in CRITICAL_FIELDS:
        if pred.get(field) != gold.get(field):
            return True

    return False

In [ ]:
@torch.no_grad()
def generate_batch(prompt_messages_batch):
    prompts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
        for msgs in prompt_messages_batch
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
        add_special_tokens=False,
    )

    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}
    prompt_width = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    outputs = []
    for i in range(generated.shape[0]):
        new_tokens = generated[i][prompt_width:]
        outputs.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return outputs

def evaluate_rows(split_name, rows):
    report_path = REPORT_DIR / f"{split_name}_stratified_predictions.jsonl"
    metrics_path = REPORT_DIR / f"{split_name}_stratified_metrics.json"
    error_path = REPORT_DIR / f"{split_name}_stratified_error_examples.json"

    total = len(rows)
    counters = Counter()

    indicator_tp = indicator_fp = indicator_fn = 0
    country_tp = country_fp = country_fn = 0
    group_tp = group_fp = group_fn = 0

    confusion_intent = Counter()
    confusion_family = Counter()
    schema_error_top = Counter()
    dangerous_examples = []
    schema_examples = []
    mismatch_examples = []

    start_time = time.time()

    with report_path.open("w", encoding="utf-8") as f_out:
        for start in range(0, total, EVAL_BATCH_SIZE):
            batch = rows[start:start + EVAL_BATCH_SIZE]

            prompt_batch = []
            gold_batch = []

            for row in batch:
                messages = row.get("messages")
                if messages and len(messages) >= 2:
                    prompt = messages[:2]
                else:
                    prompt = [
                        {"role": "system", "content": "You are a semantic parser for a Government Economic Indicator AI Agent. Output only valid JSON. Do not answer the question."},
                        {"role": "user", "content": row["user_message"]},
                    ]

                gold = row.get("assistant_json")
                if gold is None and messages and len(messages) >= 3:
                    gold = json.loads(messages[2]["content"])

                prompt_batch.append(prompt)
                gold_batch.append(gold)

            decoded_batch = generate_batch(prompt_batch)

            for row, gold, decoded in zip(batch, gold_batch, decoded_batch):
                pred = None
                json_ok = False
                schema_ok = False
                errors = []

                try:
                    pred = safe_json_loads(decoded)
                    json_ok = True
                    counters["valid_json"] += 1
                except Exception as e:
                    errors.append(f"json_error:{str(e)}")

                if pred is not None:
                    val_errors = validate_parsed_query(pred)
                    if not val_errors:
                        schema_ok = True
                        counters["schema_pass"] += 1
                    else:
                        errors.extend(val_errors)
                        for e in val_errors:
                            schema_error_top[e] += 1

                    if pred == gold:
                        counters["exact_full_json"] += 1

                    if schema_ok:
                        if pred.get("intent") == gold.get("intent"):
                            counters["intent_correct"] += 1
                        else:
                            confusion_intent[(gold.get("intent"), pred.get("intent"))] += 1

                        if pred.get("question_family") == gold.get("question_family"):
                            counters["family_correct"] += 1
                        else:
                            confusion_family[(gold.get("question_family"), pred.get("question_family"))] += 1

                        if pred.get("start_year") == gold.get("start_year") and pred.get("end_year") == gold.get("end_year"):
                            counters["year_correct"] += 1

                        for field in ["ranking_order", "aggregation", "chart_preference", "needs_clarification"]:
                            if pred.get(field) == gold.get(field):
                                counters[f"{field}_correct"] += 1

                        tp, fp, fn = list_micro_counts(pred.get("indicators"), gold.get("indicators"))
                        indicator_tp += tp; indicator_fp += fp; indicator_fn += fn

                        tp, fp, fn = list_micro_counts(pred.get("countries"), gold.get("countries"))
                        country_tp += tp; country_fp += fp; country_fn += fn

                        tp, fp, fn = list_micro_counts(pred.get("country_groups"), gold.get("country_groups"))
                        group_tp += tp; group_fp += fp; group_fn += fn

                    if is_executable(pred):
                        counters["executable_plan"] += 1

                    dangerous = is_dangerous_wrong(pred, gold, schema_ok)
                    if dangerous:
                        counters["dangerous_wrong"] += 1
                        if len(dangerous_examples) < 50:
                            dangerous_examples.append({
                                "sample_id": row.get("sample_id"),
                                "generation_source": row.get("generation_source"),
                                "language_style": row.get("language_style"),
                                "user_message": row.get("user_message"),
                                "gold": gold,
                                "pred": pred,
                                "decoded": decoded,
                            })

                    if errors and len(schema_examples) < 30:
                        schema_examples.append({
                            "sample_id": row.get("sample_id"),
                            "user_message": row.get("user_message"),
                            "errors": errors,
                            "decoded": decoded,
                            "gold": gold,
                        })

                    if schema_ok and pred != gold and len(mismatch_examples) < 50:
                        mismatch_examples.append({
                            "sample_id": row.get("sample_id"),
                            "generation_source": row.get("generation_source"),
                            "language_style": row.get("language_style"),
                            "user_message": row.get("user_message"),
                            "gold": gold,
                            "pred": pred,
                        })

                f_out.write(json.dumps({
                    "sample_id": row.get("sample_id"),
                    "generation_source": row.get("generation_source"),
                    "language_style": row.get("language_style"),
                    "user_message": row.get("user_message"),
                    "gold": gold,
                    "decoded": decoded,
                    "pred": pred,
                    "json_ok": json_ok,
                    "schema_ok": schema_ok,
                    "errors": errors,
                }, ensure_ascii=False) + "\n")

            done = min(start + EVAL_BATCH_SIZE, total)
            if done % 100 == 0 or done == total:
                elapsed = time.time() - start_time
                speed = done / elapsed if elapsed else 0
                eta = (total - done) / speed if speed else 0
                print(f"{split_name}: {done}/{total} | {speed:.3f} samples/s | ETA {eta/60:.1f}m", flush=True)

    def rate(x):
        return x / total if total else 0

    schema_total = counters["schema_pass"] or 1

    def prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 1.0
        r = tp / (tp + fn) if (tp + fn) else 1.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return {"precision": p, "recall": r, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

    metrics = {
        "split": split_name,
        "total": total,
        "valid_json_rate": rate(counters["valid_json"]),
        "schema_pass_rate": rate(counters["schema_pass"]),
        "exact_full_json_rate": rate(counters["exact_full_json"]),
        "intent_accuracy_on_schema_pass": counters["intent_correct"] / schema_total,
        "question_family_accuracy_on_schema_pass": counters["family_correct"] / schema_total,
        "year_exact_accuracy_on_schema_pass": counters["year_correct"] / schema_total,
        "ranking_order_accuracy_on_schema_pass": counters["ranking_order_correct"] / schema_total,
        "aggregation_accuracy_on_schema_pass": counters["aggregation_correct"] / schema_total,
        "chart_preference_accuracy_on_schema_pass": counters["chart_preference_correct"] / schema_total,
        "needs_clarification_accuracy_on_schema_pass": counters["needs_clarification_correct"] / schema_total,
        "indicator_micro": prf(indicator_tp, indicator_fp, indicator_fn),
        "country_micro": prf(country_tp, country_fp, country_fn),
        "country_group_micro": prf(group_tp, group_fp, group_fn),
        "executable_plan_rate": rate(counters["executable_plan"]),
        "dangerous_wrong_query_rate": rate(counters["dangerous_wrong"]),
        "dangerous_wrong_query_count": counters["dangerous_wrong"],
        "schema_error_top": dict(schema_error_top.most_common(30)),
        "intent_confusion_top": [
            {"gold": k[0], "pred": k[1], "count": v}
            for k, v in confusion_intent.most_common(30)
        ],
        "family_confusion_top": [
            {"gold": k[0], "pred": k[1], "count": v}
            for k, v in confusion_family.most_common(30)
        ],
        "runtime_seconds": round(time.time() - start_time, 2),
        "prediction_path": str(report_path),
        "metrics_path": str(metrics_path),
        "error_examples_path": str(error_path),
    }

    error_payload = {
        "dangerous_examples": dangerous_examples,
        "schema_examples": schema_examples,
        "mismatch_examples": mismatch_examples,
    }

    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    with error_path.open("w", encoding="utf-8") as f:
        json.dump(error_payload, f, ensure_ascii=False, indent=2)

    print("Saved:", metrics_path, flush=True)
    print(json.dumps(metrics, ensure_ascii=False, indent=2), flush=True)
    return metrics

validation_metrics = evaluate_rows("validation", eval_validation_rows)
test_metrics = evaluate_rows("test", eval_test_rows)

In [ ]:
SMOKE_MESSAGES = [
    "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023.",
    "Top 10 nước có lạm phát CPI cao nhất năm 2020.",
    "Xu hướng thất nghiệp của Nhật Bản sau COVID.",
    "Việt Nam có dữ liệu nợ công từ năm nào đến năm nào?",
    "Vẽ line chart cho tăng trưởng GDP thực của Indonesia giai đoạn 2000-2022.",
    "Nước nào trong ASEAN có tỷ lệ nghèo giảm mạnh nhất?",
    "Dự báo nợ công Việt Nam bằng ARIMA.",
    "Viết SQL lấy bảng gold_fiscal_monetary.",
    "Việt Nam thế nào?",
    "So sánh Việt Nam với Thái Lan.",
]

SYSTEM_PROMPT = "You are a semantic parser for a Government Economic Indicator AI Agent. Output only valid JSON. Do not answer the question."

smoke_results = []
for msg in SMOKE_MESSAGES:
    decoded = generate_batch([[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": msg}]])[0]
    pred = None
    valid_json = False
    schema_errors = []
    try:
        pred = safe_json_loads(decoded)
        valid_json = True
        schema_errors = validate_parsed_query(pred)
    except Exception as e:
        schema_errors = [str(e)]

    smoke_results.append({
        "message": msg,
        "raw_output": decoded,
        "pred": pred,
        "valid_json": valid_json,
        "schema_pass": valid_json and not schema_errors,
        "schema_errors": schema_errors,
    })

smoke_path = REPORT_DIR / "manual_smoke_test_results.json"
with smoke_path.open("w", encoding="utf-8") as f:
    json.dump(smoke_results, f, ensure_ascii=False, indent=2)

print("Saved:", smoke_path, flush=True)
print(json.dumps(smoke_results, ensure_ascii=False, indent=2), flush=True)

In [ ]:
summary = {
    "base_model_name": BASE_MODEL_NAME,
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "eval_max_samples_per_split": EVAL_MAX_SAMPLES_PER_SPLIT,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
}

summary_path = REPORT_DIR / "eval_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

zip_path = EXPORT_DIR / "parser_eval_reports.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in REPORT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(REPORT_DIR))

print("REPORT_DIR:", REPORT_DIR, flush=True)
print("Eval report zip:", zip_path, flush=True)
print("Zip size MB:", round(zip_path.stat().st_size / 1024**2, 2), flush=True)

print("\nFiles to send/check:")
for p in sorted(REPORT_DIR.iterdir()):
    print("-", p.name, round(p.stat().st_size / 1024**2, 2), "MB", flush=True)